# Urdu + Arabic book OCR (Qaari / Qari, one model per block)

This notebook turns a range of pages from a scanned Urdu/Arabic PDF into a clean UTF-8 `.txt` file.

**How it works**
1. Each page is rendered at 300 DPI, straightened, and cut into blocks: paragraphs, centred lines, poetry hemistichs, tables, figures, footnotes, and running headers.
2. A block is read briefly by **both** models: [Qaari 0.1 Urdu](https://huggingface.co/oddadmix/Qaari-0.1-Urdu-OCR-VL-2B-Instruct) (Nastaliq) and [Qari-OCR 0.2.2.1 Arabic](https://huggingface.co/NAMAA-Space/Qari-OCR-0.2.2.1-VL-2B-Instruct) (keeps harakat). The more convincing model then reads the whole block. If neither model is clearly better for a paragraph, each line is routed separately.
3. Words the model was unsure of get a **`[?]`** mark. These come from the model's own token probabilities, not from a prompt.
4. Every page is saved to Drive as soon as it is done. If Colab disconnects, run the cells again and it resumes where it stopped.
5. Pages with problems go to a review queue. Problems include tables, figures, many `[?]` marks, repetition loops, cut-off output, and blocks whose script was unclear. You can correct those pages in the notebook before exporting.

**Before you start:** open *Runtime ▸ Change runtime type* and pick a GPU. **L4** is a good choice on Colab Pro, and **T4** also works. Then run the cells in order. Do step 5 (one test page) before running a whole chapter.

**Known limits:** both models were trained on clean, computer-typeset text. The Urdu model saw 5 Nastaliq fonts and the Arabic model saw 12 fonts. Old kitabat or calligraphy, heavy bleed-through, and very small print will read worse. An Arabic phrase *inside* an Urdu line is read by the Urdu model, so its harakat may be lost; such phrases are usually marked `[?]`. Tables without ruling lines are read as ordinary text.

In [ ]:
#@title 1 · Settings { display-mode: "form" }
#@markdown **Book and chapter**
PDF_PATH = "/content/drive/MyDrive/OCR/book.pdf"  #@param {type:"string"}
CHAP_NAME = "Chapter_01"  #@param {type:"string"}
START_PAGE = 13  #@param {type:"integer"}
END_PAGE = 15  #@param {type:"integer"}
OUTPUT_FOLDER = "/content/drive/MyDrive/OCR"  #@param {type:"string"}
#@markdown PDF page numbers (the first page of the file is 1), not the numbers printed on the pages.

#@markdown ---
#@markdown **Reading**
SCRIPT_ROUTING = "auto"  #@param ["auto", "urdu", "arabic"]
#@markdown `auto` = choose Urdu or Arabic model per block. Use `urdu` / `arabic` only if the whole range is one language (faster).
FLAG_BELOW_CONFIDENCE = 0.5  #@param {type:"slider", min:0, max:0.95, step:0.05}
#@markdown A word gets `[?]` when the model's confidence in any part of it is below this. 0 = no marks.
TEXT_LINE_HEIGHT_PX = 64  #@param [48, 64, 88] {type:"raw"}
#@markdown Size each text line is scaled to before reading. Step 5 shows which works best for your book.
COLUMNS = 1  #@param [1, 2] {type:"raw"}
POETRY_FORMAT = "lines"  #@param ["lines", "tab"]
#@markdown `lines` = each hemistich on its own line, blank line between couplets. `tab` = one couplet per line.
PAGE_MARKERS = True  #@param {type:"boolean"}
#@markdown Adds `[p. 12 | printed 245]` before each page in the output.

#@markdown ---
#@markdown **Advanced**
REDO_PAGES = ""  #@param {type:"string"}
#@markdown Pages to OCR again even though they are saved, e.g. `12, 15-17`. Leave empty to resume normally.
BASE_MODEL = "full precision (recommended)"  #@param ["full precision (recommended)", "4bit (less GPU memory; may not work with newer libraries)"]
BATCH_SIZE = 4  #@param {type:"integer"}
print("Settings saved.")

In [ ]:
#@title 2 · Connect Google Drive and install packages
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes", "pymupdf"], check=True)
try:
    import cv2
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless"], check=True)

import transformers
from packaging.version import Version
if Version(transformers.__version__) < Version("4.45"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.49", "accelerate"], check=True)
    print("⚠ transformers was upgraded. Now use Runtime ▸ Restart session, then run the cells again from step 1.")

if os.path.exists(PDF_PATH):
    print(f"✓ PDF found: {PDF_PATH}")
else:
    print(f"⚠ No file at {PDF_PATH} – upload the PDF to Drive and fix PDF_PATH in step 1.")

In [ ]:
#@title 3a · Code: page layout (just run)
# ============================================================================
# LAYOUT: deskew, clean-up, block / line / paragraph segmentation, reading order
# Pure OpenCV + NumPy (runs on CPU, no model needed).
# ============================================================================
import math
from dataclasses import dataclass, field

import cv2
import numpy as np
from PIL import Image


@dataclass
class LayoutConfig:
    columns: int = 1              # 1 = single column (poetry-safe); 2 = two-column pages
    hgap_ch: float = 2.0          # horizontal gap (in glyph heights) still joined into one line
    vgap_ch: float = 1.2          # vertical gap (in glyph heights) still joined into one block
    max_lines_per_unit: int = 10  # long paragraphs are OCR'd in chunks of this many lines
    target_line_px: int = 64      # every OCR crop is rescaled so a text line is ~this tall
    max_deskew_deg: float = 3.0
    detect_footnotes: bool = True
    detect_headers: bool = True


@dataclass(eq=False)
class Line:
    comps: np.ndarray
    bbox: tuple                   # x0, y0, x1, y1 (exclusive)
    kind: str = "full"            # full | start | end | center | other


@dataclass(eq=False)
class Unit:
    uid: str
    kind: str                     # prose | centered | poem_line | line | table | figure | header | footer
    region: str                   # body | footnote | header | footer
    bbox: tuple
    comps: np.ndarray
    lines: list
    group: str                    # units in one group share a single Urdu/Arabic routing decision
    meta: dict = field(default_factory=dict)


@dataclass
class PageLayout:
    gray: np.ndarray
    labels: np.ndarray
    units: list
    angle: float
    ch: float
    line_h: float
    text_box: tuple
    footnote_y: object
    notes: list


# ----------------------------------------------------------------------------- basics
def binarize(gray):
    g = cv2.GaussianBlur(gray, (3, 3), 0)
    otsu_t, _ = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    bs = max(31, (min(gray.shape) // 25) | 1)
    ad = cv2.adaptiveThreshold(g, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, bs, 12)
    return (ad > 0) & (g < min(otsu_t + 25, 235))


def estimate_skew(ink, max_deg=3.0):
    h, w = ink.shape
    s = 1000.0 / max(h, w)
    small = cv2.resize(ink.astype(np.uint8) * 255, (max(1, int(w * s)), max(1, int(h * s))),
                       interpolation=cv2.INTER_AREA)
    c = (small.shape[1] / 2.0, small.shape[0] / 2.0)

    def score(a):
        M = cv2.getRotationMatrix2D(c, a, 1.0)
        r = cv2.warpAffine(small, M, (small.shape[1], small.shape[0]), flags=cv2.INTER_NEAREST, borderValue=0)
        p = r.sum(1).astype(np.float64)
        return float(np.sum(np.diff(p) ** 2))

    coarse = np.arange(-max_deg, max_deg + 1e-9, 0.25)
    best = max(coarse, key=score)
    fine = np.arange(best - 0.25, best + 0.25 + 1e-9, 0.05)
    return float(max(fine, key=score))


def rotate_gray(gray, angle):
    if abs(angle) < 0.05:
        return gray
    h, w = gray.shape
    M = cv2.getRotationMatrix2D((w / 2.0, h / 2.0), angle, 1.0)
    return cv2.warpAffine(gray, M, (w, h), flags=cv2.INTER_LINEAR, borderValue=255)


def _weighted_median(values, weights):
    if len(values) == 0:
        return 0.0
    o = np.argsort(values)
    v, wt = values[o], weights[o].astype(np.float64)
    cw = np.cumsum(wt)
    return float(v[np.searchsorted(cw, cw[-1] / 2.0)])


def _grid_lines(mask):
    """Count long horizontal / vertical strokes inside a component mask (table detection)."""
    h, w = mask.shape
    m = mask.astype(np.uint8)
    hk = cv2.getStructuringElement(cv2.MORPH_RECT, (max(3, int(w * 0.5)), 1))
    vk = cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(3, int(h * 0.5))))
    hl = cv2.morphologyEx(m, cv2.MORPH_OPEN, hk)
    vl = cv2.morphologyEx(m, cv2.MORPH_OPEN, vk)

    def runs(profile):
        on = profile > 0
        return int(np.sum(on[1:] & ~on[:-1]) + (1 if on.size and on[0] else 0))

    return runs(hl.sum(1)), runs(vl.sum(0))


# ----------------------------------------------------------------------------- lines
def _mk_line(ids, stats):
    ids = np.asarray(ids)
    xs0 = stats[ids, 0]; ys0 = stats[ids, 1]
    return Line(ids, (int(xs0.min()), int(ys0.min()),
                      int((xs0 + stats[ids, 2]).max()), int((ys0 + stats[ids, 3]).max())))


def couplet_split(ln, stats, cents, ch, ref_w):
    """If a line is two hemistichs separated by a wide central gap, return (right, left)."""
    ids = ln.comps
    x0, x1 = ln.bbox[0], ln.bbox[2]
    w = x1 - x0
    if w < 6 * ch or ids.size < 4:
        return None
    cov = np.zeros(w + 1, bool)
    for a, b in zip(stats[ids, 0] - x0, stats[ids, 0] + stats[ids, 2] - x0):
        cov[a:b] = True
    best, run_start = (0, 0, 0), None
    for i, c in enumerate(cov):
        if not c and run_start is None:
            run_start = i
        elif c and run_start is not None:
            if i - run_start > best[0]:
                best = (i - run_start, run_start, i)
            run_start = None
    L, g0, g1 = best
    if L < max(3 * ch, 0.12 * ref_w) or not (0.25 * w < (g0 + g1) / 2 < 0.75 * w):
        return None
    mid = x0 + (g0 + g1) / 2
    r_ids, l_ids = ids[cents[ids, 0] >= mid], ids[cents[ids, 0] < mid]
    if r_ids.size == 0 or l_ids.size == 0:
        return None
    R, Lf = _mk_line(r_ids, stats), _mk_line(l_ids, stats)
    rw, lw = R.bbox[2] - R.bbox[0], Lf.bbox[2] - Lf.bbox[0]
    if not 0.5 < rw / max(1, lw) < 2.0:
        return None
    return R, Lf


def split_lines(comp_ids, stats, cents, ch):
    """Split a set of components into text lines using a smoothed horizontal projection."""
    comp_ids = np.asarray(comp_ids)
    if comp_ids.size == 0:
        return []
    x0 = stats[comp_ids, 0]; y0 = stats[comp_ids, 1]
    x1 = x0 + stats[comp_ids, 2]; y1 = y0 + stats[comp_ids, 3]
    Y0, Y1 = int(y0.min()), int(y1.max())
    H = Y1 - Y0

    def mk(ids):
        return _mk_line(ids, stats)

    if H < 1.7 * ch:
        return [mk(comp_ids)]
    prof = np.zeros(H, np.float64)
    for a, b, w in zip(y0 - Y0, y1 - Y0, stats[comp_ids, 4]):
        prof[a:b] += w / max(1, (b - a))
    k = max(3, int(ch / 3)) | 1
    prof = np.convolve(prof, np.ones(k) / k, mode="same")
    # line pitch from autocorrelation
    p = prof - prof.mean()
    ac = np.correlate(p, p, mode="full")[H - 1:]
    lo, hi = int(0.8 * ch), min(H - 1, int(3.5 * ch))
    pitch = lo + int(np.argmax(ac[lo:hi])) if hi > lo + 1 else int(1.6 * ch)
    if hi <= lo + 1 or ac[pitch] <= 0.05 * ac[0]:
        pitch = int(1.6 * ch)
    # peaks
    mind = max(2, int(0.6 * pitch))
    thr = 0.25 * prof.max()
    cand = [i for i in range(1, H - 1) if prof[i] >= thr and prof[i] >= prof[i - 1] and prof[i] >= prof[i + 1]]
    peaks = []
    for i in sorted(cand, key=lambda i: -prof[i]):
        if all(abs(i - j) >= mind for j in peaks):
            peaks.append(i)
    peaks.sort()
    if len(peaks) <= 1:
        return [mk(comp_ids)]
    cuts = [a + int(np.argmin(prof[a:b])) for a, b in zip(peaks[:-1], peaks[1:])]
    cy = cents[comp_ids, 1] - Y0
    li = np.searchsorted(np.array(cuts), cy)
    out = []
    for k in range(len(cuts) + 1):
        ids = comp_ids[li == k]
        if ids.size:
            out.append(mk(ids))
    # merge "lines" that are only marks (tiny components) into the nearest real line
    real = [ln for ln in out if stats[ln.comps, 3].max() >= 0.6 * ch]
    if not real:
        return [mk(comp_ids)]
    for ln in out:
        if ln in real:
            continue
        cyl = (ln.bbox[1] + ln.bbox[3]) / 2.0
        tgt = min(real, key=lambda r: abs((r.bbox[1] + r.bbox[3]) / 2.0 - cyl))
        merged = mk(np.concatenate([tgt.comps, ln.comps]))
        tgt.comps, tgt.bbox = merged.comps, merged.bbox
    return real


def classify_line(ln, ref_x0, ref_x1, ch):
    ref_w = ref_x1 - ref_x0
    ind_r = ref_x1 - ln.bbox[2]
    ind_l = ln.bbox[0] - ref_x0
    tol = 1.0 * ch
    if ind_r < tol and ind_l < tol:
        return "full"
    if ind_r < tol:
        return "end"
    if ind_l < tol and ind_r < 5 * ch:
        return "start"
    if abs(ind_r - ind_l) < max(1.5 * ch, 0.08 * ref_w):
        return "center"
    return "other"


# ----------------------------------------------------------------------------- main
def segment_page(gray, cfg=None):
    cfg = cfg or LayoutConfig()
    notes = []
    ink0 = binarize(gray)
    angle = estimate_skew(ink0, cfg.max_deskew_deg)
    if abs(angle) >= 0.05:
        gray = rotate_gray(gray, angle)
        ink = binarize(gray)
    else:
        ink = ink0
    H, W = ink.shape
    n, labels, stats, cents = cv2.connectedComponentsWithStats(ink.astype(np.uint8), connectivity=8)
    ids = np.arange(1, n)
    xs, ys, ws, hs, ar = (stats[1:, i] for i in range(5))
    ok = (hs >= 4) & (ar >= 12) & (hs < 0.08 * H) & (ws < 0.5 * W)
    ch = _weighted_median(hs[ok].astype(np.float64), ar[ok]) if ok.any() else 20.0
    ch = float(max(ch, 8.0))

    alive = np.zeros(n, bool); alive[1:] = True
    alive[1:] &= ar >= max(4, int((0.06 * ch) ** 2))              # speckle noise
    dens = ar / np.maximum(1, ws * hs)

    hrule_y = []
    foot_cands = []
    tables, figures = [], []
    for i in ids[alive[1:]]:
        x, y, w, h, a = stats[i]
        d = a / max(1, w * h)
        if w > 0.7 * W and h > 0.6 * H and d < 0.15:                    # page frame
            alive[i] = False; notes.append("frame removed"); continue
        if w > 8 * ch and h < 0.35 * ch and w > 20 * h:                  # horizontal rule
            alive[i] = False
            if 0.08 * W < w < 0.55 * W and y > 0.45 * H:
                foot_cands.append((y, x, w))
            elif w >= 0.55 * W:
                hrule_y.append(y)
            continue
        if h > 8 * ch and w < 0.35 * ch:                                   # vertical rule
            alive[i] = False; continue
        if w > 4 * ch and h > 2 * ch and d < 0.3:                          # boxes / tables
            nh, nv = _grid_lines(labels[y:y + h, x:x + w] == i)
            if nh >= 2 and nv >= 2:
                alive[i] = False
                if nh >= 3 or nv >= 3:
                    tables.append([x, y, x + w, y + h])
                else:
                    notes.append("box outline removed")
                continue
        if w > 5 * ch and h > 5 * ch:                                      # big artwork / logo
            alive[i] = False; figures.append([x, y, x + w, y + h]); continue

    # table cells: everything inside a table box belongs to the table
    comp_box = np.stack([stats[:, 0], stats[:, 1], stats[:, 0] + stats[:, 2], stats[:, 1] + stats[:, 3]], 1)
    owner = {}
    for kind, boxes in (("table", tables), ("figure", figures)):
        for bi, (bx0, by0, bx1, by1) in enumerate(boxes):
            inside = alive & (cents[:, 0] >= bx0) & (cents[:, 0] < bx1) & (cents[:, 1] >= by0) & (cents[:, 1] < by1)
            inside[0] = False
            for c in np.nonzero(inside)[0]:
                owner[c] = (kind, bi)
            alive &= ~inside

    # footnote separator: short rule in lower half with text below and nothing beside it
    footnote_y = None
    if cfg.detect_footnotes and foot_cands:
        for (fy, fx, fw) in sorted(foot_cands):
            beside = alive & (np.abs(cents[:, 1] - fy) < 0.5 * ch)
            below = alive & (cents[:, 1] > fy + 0.3 * ch)
            if beside.sum() <= 2 and below.sum() >= 5:
                footnote_y = int(fy); break

    # ---------------- blocks by run-length smearing
    text_ids = np.nonzero(alive)[0]
    mask = np.isin(labels, text_ids).astype(np.uint8)
    hk = max(3, int(cfg.hgap_ch * ch)); vk = max(3, int(cfg.vgap_ch * ch))
    sm = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (hk, 1)))
    sm = cv2.morphologyEx(sm, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (1, vk)))
    if footnote_y is not None:                      # never merge across the footnote rule
        sm[max(0, footnote_y - 1):footnote_y + 2, :] = 0
    nb, blab = cv2.connectedComponents(sm, connectivity=8)
    comp2block = np.full(n, -1, np.int64)
    sel = mask.astype(bool)
    comp2block[labels[sel]] = blab[sel]
    comp2block[0] = -1

    blocks = []
    for b in range(1, nb):
        cids = np.nonzero(comp2block == b)[0]
        cids = cids[alive[cids]]
        if cids.size == 0:
            continue
        bx0 = int(stats[cids, 0].min()); by0 = int(stats[cids, 1].min())
        bx1 = int((stats[cids, 0] + stats[cids, 2]).max()); by1 = int((stats[cids, 1] + stats[cids, 3]).max())
        bw, bh = bx1 - bx0, by1 - by0
        if min(bw, bh) < 0.5 * ch and max(bw, bh) > 5 * ch:
            continue                                                # ornament / stray rule
        if bh > 6 * ch and bw < 1.5 * ch:
            continue                                                # border ornament strip
        ink_d = stats[cids, 4].sum() / max(1, bw * bh)
        if ink_d > 0.42 and bw > 3 * ch and bh > 3 * ch:
            figures.append([bx0, by0, bx1, by1]); continue
        blocks.append({"kind": "text", "bbox": (bx0, by0, bx1, by1), "comps": cids})
    for bi, bb in enumerate(tables):
        cids = np.array([c for c, o in owner.items() if o == ("table", bi)], np.int64)
        blocks.append({"kind": "table", "bbox": tuple(int(v) for v in bb), "comps": cids})
    for bb in figures:
        blocks.append({"kind": "figure", "bbox": tuple(int(v) for v in bb), "comps": np.zeros(0, np.int64)})

    for b in blocks:
        y0, y1 = b["bbox"][1], b["bbox"][3]
        b["region"] = "footnote" if (footnote_y is not None and y0 >= footnote_y) else "body"
        if b["kind"] == "text":
            c = b["comps"]
            big = stats[c, 3] >= 4
            bch = _weighted_median(stats[c, 3][big].astype(np.float64), stats[c, 4][big]) if big.any() else ch
            b["ch"] = float(np.clip(bch, 0.4 * ch, 2.5 * ch))
            b["lines"] = split_lines(c, stats, cents, b["ch"])
        else:
            b["lines"] = []

    text_blocks = [b for b in blocks if b["kind"] == "text"]
    wide = [b for b in text_blocks if (b["bbox"][2] - b["bbox"][0]) > 0.3 * W] or text_blocks
    if wide:
        tb = (min(b["bbox"][0] for b in wide), min(b["bbox"][1] for b in text_blocks),
              max(b["bbox"][2] for b in wide), max(b["bbox"][3] for b in text_blocks))
    else:
        tb = (0, 0, W, H)
    lh_all = [ln.bbox[3] - ln.bbox[1] for b in text_blocks for ln in b["lines"]
              if (ln.bbox[2] - ln.bbox[0]) > 4 * ch]
    line_h = float(np.median(lh_all)) if lh_all else 1.3 * ch

    # ---------------- running header / footer candidates (confirmed later from OCR text)
    if cfg.detect_headers and text_blocks:
        body = sorted([b for b in text_blocks if b["region"] == "body"], key=lambda b: b["bbox"][1])
        if body:
            top = body[0]
            band = [b for b in body if b["bbox"][1] < top["bbox"][3]]
            rest = [b for b in body if b not in band]
            gap_ok = (not rest) or (min(b["bbox"][1] for b in rest) - max(b["bbox"][3] for b in band) > 0.6 * ch)
            rule_ok = any(0 <= y - max(b["bbox"][3] for b in band) < 2.0 * ch for y in hrule_y)
            if (max(b["bbox"][3] for b in band) < 0.12 * H and all(len(b["lines"]) == 1 for b in band)
                    and rest and (gap_ok or rule_ok)):
                for b in band:
                    b["region"] = "header"
        for b in text_blocks:
            bw = b["bbox"][2] - b["bbox"][0]
            if b["bbox"][1] > 0.9 * H and len(b["lines"]) == 1 and bw < 0.3 * (tb[2] - tb[0]):
                b["region"] = "footer"

    # ---------------- reading order + units
    units = []
    uid = [0]
    ref_cache = {}

    def ref_for(b):
        if cfg.columns == 2 and b.get("col") is not None:
            return ref_cache[b["col"]]
        return tb[0], tb[2]

    # optional two-column split
    if cfg.columns == 2 and text_blocks:
        cx0, cx1 = tb[0], tb[2]
        body_mask = np.isin(labels, np.concatenate([b["comps"] for b in text_blocks if b["region"] == "body"]
                                                   or [np.zeros(0, np.int64)]))
        prof = body_mask[:, cx0:cx1].sum(0).astype(np.float64)
        a, bnd = int(0.3 * (cx1 - cx0)), int(0.7 * (cx1 - cx0))
        gx = cx0 + a + int(np.argmin(np.convolve(prof, np.ones(9) / 9, "same")[a:bnd])) if bnd > a else (cx0 + cx1) // 2
        ref_cache["R"] = (gx, cx1); ref_cache["L"] = (cx0, gx)
        for b in blocks:
            x0, _, x1, _ = b["bbox"]
            b["col"] = "R" if x0 >= gx - ch else ("L" if x1 <= gx + ch else None)
        notes.append(f"2-column split at x={gx}")

    gid = [0]

    def new_uid():
        uid[0] += 1
        return f"u{uid[0]:03d}"

    def new_gid(prefix="g"):
        gid[0] += 1
        return f"{prefix}{gid[0]}"

    def emit_text_block(b, group=None):
        region = b["region"]
        rx0, rx1 = ref_for(b)
        lines = b["lines"]
        for ln in lines:
            ln.kind = classify_line(ln, rx0, rx1, ch)
        if region in ("header", "footer"):
            u = Unit(new_uid(), region, region, b["bbox"], b["comps"], lines, group or new_gid())
            units.append(u); return
        cur, cur_kind = [], None

        def flush():
            nonlocal cur, cur_kind
            if not cur:
                return
            chunks = [cur[i:i + cfg.max_lines_per_unit] for i in range(0, len(cur), cfg.max_lines_per_unit)]
            g = group or new_gid()
            for ci, ch_lines in enumerate(chunks):
                comps = np.concatenate([l.comps for l in ch_lines])
                bb = (min(l.bbox[0] for l in ch_lines), min(l.bbox[1] for l in ch_lines),
                      max(l.bbox[2] for l in ch_lines), max(l.bbox[3] for l in ch_lines))
                units.append(Unit(new_uid(), cur_kind, region, bb, comps, ch_lines, g,
                                  {"continues": ci < len(chunks) - 1}))
            cur, cur_kind = [], None

        poem_g = None
        for ln in lines:
            cs = couplet_split(ln, stats, cents, ch, rx1 - rx0)
            if cs:
                flush()
                if poem_g is None:
                    poem_g = new_gid("poem")
                for half, hl in zip("RL", cs):
                    hl.kind = "full"
                    units.append(Unit(new_uid(), "poem_line", region, hl.bbox, hl.comps, [hl], poem_g,
                                      {"half": half, "couplet_end": half == "L"}))
                continue
            poem_g = None
            k = "prose" if ln.kind in ("full", "start", "end") else "centered"
            if k != cur_kind or (k == "prose" and ln.kind == "start"):
                flush()
            cur_kind = k
            cur.append(ln)
            if k == "prose" and ln.kind == "end":
                flush()
        flush()

    def emit_block(b):
        if b["kind"] == "text":
            emit_text_block(b)
        else:
            units.append(Unit(new_uid(), b["kind"], b["region"], b["bbox"], b["comps"], [], new_gid()))

    def emit_band(band, poem_state):
        band = sorted(band, key=lambda b: -b["bbox"][2])        # right to left
        tx = [b for b in band if b["kind"] == "text"]
        if len(band) == 2 and len(tx) == 2:
            R, L = tx
            rw, lw = R["bbox"][2] - R["bbox"][0], L["bbox"][2] - L["bbox"][0]
            refw = tb[2] - tb[0]
            if (rw < 0.55 * refw and lw < 0.55 * refw and 0.5 < rw / max(1, lw) < 2.0
                    and R["bbox"][0] > L["bbox"][2]):
                if poem_state["group"] is None:
                    poem_state["group"] = new_gid("poem")
                g = poem_state["group"]
                rl, ll = list(R["lines"]), list(L["lines"])
                rows = []
                for r in rl:
                    best = None
                    for l in ll:
                        ov = min(r.bbox[3], l.bbox[3]) - max(r.bbox[1], l.bbox[1])
                        if ov > 0.3 * min(r.bbox[3] - r.bbox[1], l.bbox[3] - l.bbox[1]):
                            best = l; break
                    rows.append((r, best))
                    if best is not None:
                        ll.remove(best)
                for ci, (r, l) in enumerate(rows):
                    for half, ln in (("R", r), ("L", l)):
                        if ln is None:
                            continue
                        units.append(Unit(new_uid(), "poem_line", R["region"], ln.bbox, ln.comps, [ln], g,
                                          {"half": half, "couplet_end": half == "L" or l is None}))
                for l in ll:
                    units.append(Unit(new_uid(), "poem_line", L["region"], l.bbox, l.comps, [l], g,
                                      {"half": "L", "couplet_end": True}))
                return True
        poem_state["group"] = None
        for b in band:
            emit_block(b)
        return False

    def bands_of(bl):
        bl = sorted(bl, key=lambda b: b["bbox"][1])
        out = []
        for b in bl:
            placed = False
            for band in out:
                by0 = min(x["bbox"][1] for x in band); by1 = max(x["bbox"][3] for x in band)
                ov = min(by1, b["bbox"][3]) - max(by0, b["bbox"][1])
                if ov > 0.5 * min(by1 - by0, b["bbox"][3] - b["bbox"][1]):
                    band.append(b); placed = True; break
            if not placed:
                out.append([b])
        return out

    def emit_region(bl):
        if cfg.columns == 2:
            seq, buf = [], {"R": [], "L": []}
            for b in sorted(bl, key=lambda b: b["bbox"][1]):
                if b.get("col") in ("R", "L"):
                    buf[b["col"]].append(b)
                else:
                    seq += buf["R"] + buf["L"]; buf = {"R": [], "L": []}
                    seq.append(b)
            seq += buf["R"] + buf["L"]
            for b in seq:
                emit_block(b)
            return
        poem_state = {"group": None}
        for band in bands_of(bl):
            emit_band(band, poem_state)

    for region in ("header", "body", "footnote", "footer"):
        emit_region([b for b in blocks if b["region"] == region])

    return PageLayout(gray=gray, labels=labels, units=units, angle=angle, ch=ch, line_h=line_h,
                      text_box=tb, footnote_y=footnote_y, notes=notes)


# ----------------------------------------------------------------------------- crops
def unit_line_height(layout, unit):
    hs = [l.bbox[3] - l.bbox[1] for l in unit.lines if (l.bbox[2] - l.bbox[0]) > 2 * layout.ch]
    return float(np.median(hs)) if hs else layout.line_h


def render_unit(layout, unit, cfg=None, masked=True):
    """Crop a unit (only its own ink), pad it, and rescale to the target line height."""
    cfg = cfg or LayoutConfig()
    H, W = layout.gray.shape
    pad = int(max(8, 0.6 * layout.ch))
    x0, y0, x1, y1 = unit.bbox
    x0, y0 = max(0, x0 - pad), max(0, y0 - pad)
    x1, y1 = min(W, x1 + pad), min(H, y1 + pad)
    crop = layout.gray[y0:y1, x0:x1]
    if masked and unit.kind not in ("table", "figure") and len(unit.comps):
        m = np.isin(layout.labels[y0:y1, x0:x1], unit.comps).astype(np.uint8)
        m = cv2.dilate(m, np.ones((3, 3), np.uint8), iterations=1)
        crop = np.where(m > 0, crop, 255).astype(np.uint8)
    crop = cv2.copyMakeBorder(crop, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=255)
    if unit.kind != "figure":
        s = float(np.clip(cfg.target_line_px / max(1.0, unit_line_height(layout, unit)), 0.3, 3.0))
        if abs(s - 1) > 0.05:
            crop = cv2.resize(crop, (max(1, int(crop.shape[1] * s)), max(1, int(crop.shape[0] * s))),
                              interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC)
    return Image.fromarray(crop).convert("RGB")


def line_units(unit):
    """Split a multi-line unit into one unit per line (used for mixed-script paragraphs)."""
    out = []
    for i, ln in enumerate(unit.lines):
        out.append(Unit(f"{unit.uid}.{i + 1}", "line", unit.region, ln.bbox, ln.comps, [ln],
                        f"{unit.group}.{i + 1}", {"parent": unit.uid, "parent_kind": unit.kind,
                                                  "last": i == len(unit.lines) - 1}))
    return out


UNIT_COLORS = {"prose": (46, 139, 87), "centered": (0, 128, 128), "poem_line": (199, 21, 133),
               "line": (46, 139, 87), "table": (255, 140, 0), "figure": (128, 128, 128),
               "header": (160, 160, 160), "footer": (160, 160, 160)}
SCRIPT_COLORS = {"urdu": (22, 163, 74), "arabic": (37, 99, 235)}


def draw_layout(layout, scripts=None, width=900):
    """Page thumbnail with numbered unit boxes (colour = script if known, else unit kind)."""
    img = cv2.cvtColor(layout.gray, cv2.COLOR_GRAY2BGR)
    s = width / img.shape[1]
    img = cv2.resize(img, (width, int(img.shape[0] * s)), interpolation=cv2.INTER_AREA)
    for i, u in enumerate(layout.units):
        col = UNIT_COLORS.get(u.kind, (0, 0, 0))
        if scripts and scripts.get(u.uid) in SCRIPT_COLORS and u.kind not in ("table", "figure"):
            col = SCRIPT_COLORS[scripts[u.uid]]
        x0, y0, x1, y1 = (int(v * s) for v in u.bbox)
        cv2.rectangle(img, (x0, y0), (x1, y1), col[::-1], 2)
        cv2.putText(img, str(i + 1), (x1 + 2, y0 + 12), cv2.FONT_HERSHEY_SIMPLEX, 0.45, col[::-1], 1, cv2.LINE_AA)
    if layout.footnote_y is not None:
        y = int(layout.footnote_y * s)
        cv2.line(img, (0, y), (width, y), (128, 0, 128), 1)
    return Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

In [ ]:
#@title 3b · Code: models and routing (just run)
# ============================================================================
# ENGINE: one Qwen2-VL-2B base + two LoRA adapters (Urdu / Arabic), batched greedy
# generation with per-token probabilities, Urdu-vs-Arabic routing, loop detection.
# ============================================================================
import codecs
import os
import re
import unicodedata

import numpy as np
from PIL import Image

PROMPT = ("Below is the image of one page of a document, as well as some raw textual content that was "
          "previously extracted for it. Just return the plain text representation of this document as if "
          "you were reading it naturally. Do not hallucinate.")   # the exact prompt both models were trained on

ADAPTERS = {"urdu": "oddadmix/Qaari-0.1-Urdu-OCR-VL-2B-Instruct",
            "arabic": "NAMAA-Space/Qari-OCR-0.2.2.1-VL-2B-Instruct"}
BASE_4BIT = "unsloth/Qwen2-VL-2B-Instruct-unsloth-bnb-4bit"   # the base both adapters were trained on
BASE_FP16 = "Qwen/Qwen2-VL-2B-Instruct"

# ----------------------------------------------------------------------------- script features
URDU_ONLY = set("\u0679\u0688\u0691\u06BA\u06D2\u06D3")        # ٹ ڈ ڑ ں ے ۓ  (never in Arabic)
HARAKAT_RE = re.compile("[\u064B-\u0652\u0670]")
LETTER_RE = re.compile("[\u0620-\u064A\u066E-\u06D3\u06FA-\u06FF]")


def n_letters(t):
    return len(LETTER_RE.findall(t))


def urdu_rate(t):
    n = n_letters(t)
    return sum(c in URDU_ONLY for c in t) / n if n else 0.0


def harakat_rate(t):
    n = n_letters(t)
    return len(HARAKAT_RE.findall(t)) / n if n else 0.0


def decide_script(u, a):
    """u / a: probe results from the Urdu / Arabic adapter on the same crop.
    Returns (script, margin, why).  Urdu-only letters (retroflex te/dal/re, noon ghunna, bari ye) never
    occur in Arabic, and a fully-vowelled text is a strong Arabic signal; otherwise the more confident
    model wins."""
    su, sa = u["mean_lp"], a["mean_lp"]
    ur_u, ur_a, hr_a = urdu_rate(u["text"]), urdu_rate(a["text"]), harakat_rate(a["text"])
    lu = n_letters(u["text"])
    score_u = su + (1.0 if ur_u >= 0.02 else 0.0) + (0.5 if ur_a >= 0.02 else 0.0)
    score_a = sa + (1.0 if hr_a >= 0.15 else 0.0) + (0.5 if (lu >= 25 and ur_u == 0) else 0.0)
    script = "urdu" if score_u >= score_a else "arabic"
    why = f"U lp={su:.2f} urdu-letters={ur_u:.0%} | A lp={sa:.2f} harakat={hr_a:.0%}"
    return script, abs(score_u - score_a), why


# ----------------------------------------------------------------------------- loops
_LOOP_RE = re.compile(r"(.{4,120}?)(?:\s*\1){3,}", re.S)


def find_loop(text):
    """Return (start, end, unit) of the first run where a chunk repeats 4+ times in a row, else None."""
    m = _LOOP_RE.search(text)
    if not m or len(m.group(0)) < 40:
        return None
    return m.start(), m.end(), m.group(1)


def trim_loop(text):
    hit = find_loop(text)
    if not hit:
        return text, False
    s, e, unit = hit
    return text[:s] + unit + " [?…]" + text[e:], True


# ----------------------------------------------------------------------------- confidence
def _bytes_decoder():
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b); cs.append(256 + n); n += 1
    return {chr(c): b for b, c in zip(bs, cs)}


_BYTE_DEC = _bytes_decoder()


def token_char_spans(tokenizer, ids, text):
    """Map each generated token to the character span it produced in `text` (byte-level BPE).
    Returns a list of (char_start, char_end) per token, or None if the mapping can't be verified."""
    toks = tokenizer.convert_ids_to_tokens([int(i) for i in ids])
    special = set(getattr(tokenizer, "all_special_ids", []) or [])
    tb = []
    for i, t in zip(ids, toks):
        if int(i) in special or t is None:
            tb.append(b"")
            continue
        try:
            tb.append(bytes(_BYTE_DEC[c] for c in t))
        except KeyError:
            return None
    raw = b"".join(tb)
    dec = codecs.getincrementaldecoder("utf-8")("replace")
    char_end_byte, pos, out = [], 0, []
    for b in raw:
        pos += 1
        s = dec.decode(bytes([b]))
        for _ in s:
            char_end_byte.append(pos)
        out.append(s)
    s = dec.decode(b"", final=True)
    for _ in s:
        char_end_byte.append(pos)
    out.append(s)
    if "".join(out) != text:
        return None
    # token byte ranges -> char ranges
    spans, b0, ci = [], 0, 0
    for t in tb:
        b1 = b0 + len(t)
        c0 = ci
        while ci < len(char_end_byte) and char_end_byte[ci] <= b1:
            ci += 1
        spans.append((c0, ci))
        b0 = b1
    return spans


def flag_uncertain(text, spans, logprobs, threshold, mark="[?]"):
    """Append `mark` to every word whose least-confident token has probability < threshold."""
    if spans is None or threshold <= 0:
        return text, 0, []
    char_p = np.ones(len(text))
    for (c0, c1), lp in zip(spans, logprobs):
        p = float(np.exp(lp))
        if c1 > c0:
            char_p[c0:c1] = np.minimum(char_p[c0:c1], p)
        elif c0 < len(text):             # token completed no char (partial UTF-8) -> charge next char
            char_p[c0] = min(char_p[c0], p)
    out, last, n, words = [], 0, 0, []
    for m in re.finditer(r"\S+", text):
        w0, w1 = m.span()
        if not LETTER_RE.search(m.group(0)) and not re.search(r"\w", m.group(0)):
            continue
        pmin = float(char_p[w0:w1].min())
        words.append((m.group(0), pmin))
        if pmin < threshold:
            # put the mark after the word but before trailing punctuation
            core_end = w1
            while core_end > w0 and unicodedata.category(text[core_end - 1])[0] == "P":
                core_end -= 1
            if core_end == w0:
                core_end = w1
            out.append(text[last:core_end]); out.append(mark)
            last = core_end
            n += 1
    out.append(text[last:])
    return "".join(out), n, words


# ----------------------------------------------------------------------------- adapters
# A small self-contained LoRA implementation (no PEFT), so library version clashes on Colab
# (bitsandbytes / torchao / PEFT / transformers) cannot break model loading.
def _torch():
    import torch
    return torch


class _LoRALinear(_torch().nn.Module):
    def __init__(self, base):
        super().__init__()
        nn = _torch().nn
        self.base_layer = base
        self.lora_A, self.lora_B = nn.ModuleDict(), nn.ModuleDict()
        self.scaling = {}
        self.active = None
        self.in_features, self.out_features = base.in_features, base.out_features

    def add(self, name, r, alpha, dtype, device, rslora=False):
        nn = _torch().nn
        self.lora_A[name] = nn.Linear(self.in_features, r, bias=False, dtype=dtype, device=device)
        self.lora_B[name] = nn.Linear(r, self.out_features, bias=False, dtype=dtype, device=device)
        self.scaling[name] = alpha / (r ** 0.5 if rslora else r)

    def forward(self, x, *args, **kwargs):
        y = self.base_layer(x, *args, **kwargs)
        a = self.active
        if a is not None and a in self.lora_A:
            A, B = self.lora_A[a], self.lora_B[a]
            y = y + (B(A(x.to(A.weight.dtype))) * self.scaling[a]).to(y.dtype)
        return y


def set_lora_adapter(model, name):
    """Activate one adapter on every LoRA layer (None = plain base model)."""
    for m in model.modules():
        if isinstance(m, _LoRALinear):
            m.active = name


def _read_adapter(src):
    import json
    if os.path.isdir(src):
        cpath, wpath = os.path.join(src, "adapter_config.json"), os.path.join(src, "adapter_model.safetensors")
    else:
        from huggingface_hub import hf_hub_download
        cpath, wpath = hf_hub_download(src, "adapter_config.json"), hf_hub_download(src, "adapter_model.safetensors")
    with open(cpath, encoding="utf-8") as f:
        return json.load(f), wpath


def load_lora_adapters(model, adapters, log=print):
    """Wrap the target Linear layers and copy each adapter's saved weights in by module-path suffix.
    This works whichever Qwen2-VL module layout the installed transformers uses (the adapters were
    saved with the older `model.layers.* / visual.*` names). Raises if anything is missed."""
    torch = _torch()
    from safetensors.torch import load_file

    pdt = next(p.dtype for p in model.parameters() if p.dtype.is_floating_point)
    report = {}
    for name, src in adapters.items():
        cfg, wpath = _read_adapter(src)
        r, alpha = int(cfg["r"]), float(cfg.get("lora_alpha", cfg["r"]))
        tm = cfg["target_modules"]
        if isinstance(tm, str):
            rx = re.compile(tm)
            is_target = lambda n: rx.fullmatch(n) is not None
        else:
            is_target = lambda n: any(n == t or n.endswith("." + t) for t in tm)
        # wrap target layers (once; a second adapter reuses the wrappers)
        wrapped = {n: m for n, m in model.named_modules() if isinstance(m, _LoRALinear)}
        for n, m in list(model.named_modules()):
            if n in wrapped or isinstance(m, _LoRALinear) or {"base_layer", "lora_A", "lora_B"} & set(n.split(".")):
                continue
            if hasattr(m, "in_features") and hasattr(m, "out_features") and hasattr(m, "weight") and is_target(n):
                parent_name, _, child = n.rpartition(".")
                parent = model.get_submodule(parent_name) if parent_name else model
                w = _LoRALinear(m)
                setattr(parent, child, w)
                wrapped[n] = w
        mods = {}
        for n, w in wrapped.items():
            if is_target(n):
                dev = w.base_layer.weight.device
                w.add(name, r, alpha, pdt, dev, rslora=bool(cfg.get("use_rslora")))
                mods[n] = w
        sd = load_file(wpath)
        index = {}
        for n in mods:
            parts = n.split(".")
            for k in range(len(parts)):
                index.setdefault(".".join(parts[k:]), []).append(n)
        loaded, unmatched = set(), []
        for key, t in sd.items():
            mm = re.match(r"(.*)\.(lora_A|lora_B)(?:\.[^.]+)?\.weight$", key)
            if not mm:
                unmatched.append(key); continue
            path, ab = mm.groups()
            parts = path.split(".")
            target = None
            for k in range(len(parts)):
                c = index.get(".".join(parts[k:]))
                if c and len(c) == 1:
                    target = c[0]; break
            if target is None:
                unmatched.append(key); continue
            wt = getattr(mods[target], ab)[name].weight
            if tuple(wt.shape) != tuple(t.shape):
                raise RuntimeError(f"{name}: shape mismatch for {key}: {tuple(t.shape)} vs {tuple(wt.shape)}")
            with torch.no_grad():
                wt.copy_(t.to(device=wt.device, dtype=wt.dtype))
            loaded.add((target, ab))
        missing = [(n, ab) for n in mods for ab in ("lora_A", "lora_B") if (n, ab) not in loaded]
        report[name] = {"lora_modules": len(mods), "tensors_in_file": len(sd), "loaded": len(loaded),
                        "missing": len(missing), "unmatched": len(unmatched)}
        if missing or unmatched or not loaded:
            raise RuntimeError(f"Adapter '{name}' did not load cleanly: {report[name]} "
                               f"e.g. missing={missing[:3]} unmatched={unmatched[:3]}")
        log(f"  adapter '{name}': {len(loaded)} LoRA tensors loaded into {len(mods)} layers ✓")
    set_lora_adapter(model, None)
    return report


# ----------------------------------------------------------------------------- engine
class OCREngine:
    def __init__(self, model, processor, max_pixels=1280 * 28 * 28, batch_size=4, log=print):
        self.model = model
        self.processor = processor
        self.tok = processor.tokenizer
        self.tok.padding_side = "left"
        self.batch_size = batch_size
        self.log = log
        ip = processor.image_processor
        for attr, val in (("max_pixels", max_pixels), ("min_pixels", 8 * 28 * 28)):
            if hasattr(ip, attr):
                setattr(ip, attr, val)
        size = getattr(ip, "size", None)
        if size is not None and (hasattr(size, "longest_edge") or (isinstance(size, dict) and "longest_edge" in size)):
            try:
                ip.size = {"shortest_edge": 8 * 28 * 28, "longest_edge": max_pixels}
            except Exception:
                pass
        self.max_pixels = max_pixels
        self.chat = processor.apply_chat_template(
            [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT}]}],
            tokenize=False, add_generation_prompt=True)
        stop = {self.tok.eos_token_id, self.tok.pad_token_id}
        for t in ("<|im_end|>", "<|endoftext|>"):
            tid = self.tok.convert_tokens_to_ids(t)
            if isinstance(tid, int) and tid >= 0:
                stop.add(tid)
        self.stop_ids = {s for s in stop if s is not None}
        self.active = None

    def use(self, adapter):
        if adapter != self.active:
            set_lora_adapter(self.model, adapter)
            self.active = adapter

    def _fit(self, im):
        """Downscale so the image never exceeds max_pixels (independent of processor version)."""
        w, h = im.size
        if w * h > self.max_pixels:
            s = (self.max_pixels / float(w * h)) ** 0.5
            im = im.resize((max(28, int(w * s)), max(28, int(h * s))), Image.LANCZOS)
        return im

    def _generate_batch(self, images, max_new_tokens, repetition_penalty=1.0):
        import torch
        images = [self._fit(im) for im in images]
        inputs = self.processor(text=[self.chat] * len(images), images=images, padding=True, return_tensors="pt")
        inputs = inputs.to(self.model.device)
        with torch.inference_mode():
            out = self.model.generate(**inputs, max_new_tokens=int(max_new_tokens), do_sample=False,
                                      num_beams=1, repetition_penalty=repetition_penalty,
                                      output_scores=True, return_dict_in_generate=True,
                                      pad_token_id=self.tok.pad_token_id)
            lps = self.model.compute_transition_scores(out.sequences, out.scores, normalize_logits=True)
        n_in = inputs["input_ids"].shape[1]
        res = []
        for i in range(len(images)):
            gen = out.sequences[i, n_in:].tolist()
            lp = lps[i].float().cpu().numpy()
            end, finished = len(gen), False
            for j, t in enumerate(gen):
                if t in self.stop_ids:
                    end, finished = j, True
                    break
            ids, lp = gen[:end], lp[:end]
            text = self.tok.decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
            lp = np.where(np.isfinite(lp), lp, -20.0)
            res.append({"text": text, "ids": ids, "lp": lp, "finished": finished,
                        "mean_lp": float(lp.mean()) if len(lp) else 0.0})
        return res

    def generate(self, images, adapter, budgets, repetition_penalty=1.0):
        """Greedy OCR of many crops with one adapter, batched (shrinks the batch on out-of-memory)."""
        import torch
        self.use(adapter)
        order = sorted(range(len(images)), key=lambda i: budgets[i])
        results = [None] * len(images)
        bs, i = self.batch_size, 0
        while i < len(order):
            idx = order[i:i + bs]
            try:
                out = self._generate_batch([images[k] for k in idx], max(budgets[k] for k in idx), repetition_penalty)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if bs == 1:
                    raise
                bs = max(1, bs // 2)
                self.log(f"    (GPU memory full – batch size reduced to {bs})")
                continue
            for k, r in zip(idx, out):
                results[k] = r
            i += len(idx)
        return results

In [ ]:
#@title 3c · Code: text clean-up (just run)
# ============================================================================
# TEXT: per-script normalisation, conservative clean-up, page assembly
# ============================================================================
import difflib
import re
import unicodedata

AR_LETTER = "\u0620-\u064A\u066E-\u06D3\u06FA-\u06FF"
AR_MARK = "\u064B-\u065F\u0670\u06D6-\u06ED"
_PROTECT = {"[?…]": "\uE001", "[?]": "\uE000"}

# ---- letter normalisation (only unambiguous mappings; nothing is deleted)
URDU_MAP = str.maketrans({"\u0643": "\u06A9",   # ك -> ک
                          "\u064A": "\u06CC",   # ي -> ی
                          "\u0649": "\u06CC",   # ى -> ی
                          **{chr(0x0660 + i): chr(0x06F0 + i) for i in range(10)}})   # ٠١٢ -> ۰۱۲
ARABIC_MAP = str.maketrans({"\u06A9": "\u0643",  # ک -> ك
                            "\u06C1": "\u0647",  # ہ -> ه
                            "\u06BE": "\u0647",  # ھ -> ه
                            "\u06C3": "\u0629",  # ۃ -> ة
                            **{chr(0x06F0 + i): chr(0x0660 + i) for i in range(10)}})  # ۰۱۲ -> ٠١٢
INVISIBLE_RE = re.compile("[\x00-\x08\x0B-\x1F\x7F\u200B\u200E\u200F\u202A-\u202E\u2066-\u2069\uFEFF]")


def normalize_letters(t, script):
    if script == "urdu":
        t = t.translate(URDU_MAP)
        t = re.sub(f"\u0647(?![{AR_LETTER}{AR_MARK}])", "\u06C1", t)       # final ه -> ہ
        t = unicodedata.normalize("NFC", t)                                  # ہ + ٔ -> ۂ , ے + ٔ -> ۓ
    elif script == "arabic":
        t = t.translate(ARABIC_MAP)
        t = re.sub(f"\u06CC(?=[{AR_MARK}]*[{AR_LETTER}])", "\u064A", t)      # medial ی -> ي
        t = t.replace("\u06CC", "\u0649")                                   # final ی -> ى
    return t


def clean_text(t, script, keep_lines=False, remove_tatweel=True):
    """Conservative clean-up. `[?]` markers are protected; harakat are never touched."""
    for k, v in _PROTECT.items():
        t = t.replace(k, v)
    t = INVISIBLE_RE.sub("", t)
    t = normalize_letters(t, script)
    if remove_tatweel:
        t = re.sub(f"\u0640+(?![{AR_MARK}])", "", t)                        # keep ـٰ in Qur'anic spelling
    ctx = f"(?<=[{AR_LETTER}{AR_MARK}\uE000\uE001])"
    t = re.sub(ctx + r"\s*,", "\u060C", t)
    t = re.sub(ctx + r"\s*;", "\u061B", t)
    t = re.sub(ctx + r"\s*\?", "\u061F", t)
    if script == "urdu":
        t = re.sub(ctx + r"\s*\.(?=\s|$)", "\u06D4", t)
    t = re.sub(r"[ \t]+([\u060C\u061B\u061F\u06D4!:.)\]»])", r"\1", t)                          # no space before punctuation
    t = re.sub(f"([\u060C\u061B\u061F\u06D4])(?=[{AR_LETTER}])", r"\1 ", t)                     # one space after
    t = re.sub(r"([(\[«])[ \t]+", r"\1", t)
    if keep_lines:
        t = "\n".join(re.sub(r"[ \t]+", " ", ln).strip() for ln in t.split("\n"))
        t = re.sub(r"\n{2,}", "\n", t).strip()
    else:
        t = re.sub(r"\s+", " ", t).strip()
    for k, v in _PROTECT.items():
        t = t.replace(v, k)
    return t


# ---- page assembly
KEEP_LINES = {"centered", "table", "page"}


def unit_text(u):
    return u.get("corrected") if u.get("corrected") is not None else u.get("text", "")


def assemble_page(rec, poem_format="lines", footnote_rule="————————", skip_regions=("header", "footer")):
    """Turn one page record into plain text: paragraphs separated by a blank line, couplets as
    two lines, centred text with its line breaks, footnotes under a short rule."""
    body, notes = [], []
    cur_para = None
    units = rec.get("units", [])
    i = 0
    while i < len(units):
        u = units[i]
        if u["region"] in skip_regions:
            i += 1; continue
        out = notes if u["region"] == "footnote" else body
        kind = u["kind"]
        if kind == "figure":
            out.append(f"[FIGURE — image saved as {u.get('file', '')}]")
        elif kind == "table":
            out.append(f"[TABLE — image saved as {u.get('file', '')}]\n" + unit_text(u))
        elif kind == "poem_line":
            lines = []
            g = u["group"]
            couplet = []
            while i < len(units) and units[i]["kind"] == "poem_line" and units[i]["group"] == g:
                couplet.append(unit_text(units[i]))
                if units[i]["meta"].get("couplet_end"):
                    lines.append(couplet); couplet = []
                i += 1
            if couplet:
                lines.append(couplet)
            if poem_format == "tab":
                out.append("\n".join("\t".join(c) for c in lines))
            else:
                out.extend("\n".join(c) for c in lines)
            continue
        elif kind == "line":                                    # a paragraph re-split for mixed script
            parts, sep = [], "\n" if u["meta"].get("parent_kind") in KEEP_LINES else " "
            parent = u["meta"].get("parent")
            while i < len(units) and units[i]["kind"] == "line" and units[i]["meta"].get("parent") == parent:
                parts.append(unit_text(units[i])); i += 1
            txt = sep.join(p for p in parts if p)
            if cur_para is not None:
                out[-1] += " " + txt
            else:
                out.append(txt)
            cur_para = None
            continue
        else:
            txt = unit_text(u)
            if cur_para is not None and cur_para is out:
                out[-1] += " " + txt
            else:
                out.append(txt)
        cur_para = out if u["meta"].get("continues") else None
        i += 1
    text = "\n\n".join(p for p in body if p.strip())
    if notes:
        text += "\n\n" + footnote_rule + "\n" + "\n".join(n for n in notes if n.strip())
    return text


# ---- running headers / page numbers (decided across pages, after OCR)
_DIGITS = "0-9\u0660-\u0669\u06F0-\u06F9"


def _strip_digits(t):
    return re.sub(f"[{_DIGITS}\\s\\W]+", " ", t).strip()


def resolve_headers(records):
    """A header/footer candidate is dropped as a running head if it is only a page number or if
    similar text appears in a header/footer within 2 pages; otherwise it is kept as body text."""
    cands = {}
    for r in records:
        cands[r["page"]] = [u for u in r.get("units", []) if u["region"] in ("header", "footer")]
    info = {}
    for r in records:
        p = r["page"]
        printed, keep = None, []
        for u in cands[p]:
            t = unit_text(u).strip()
            core = _strip_digits(t)
            digits = re.findall(f"[{_DIGITS}]+", t)
            if not core:
                if digits:
                    printed = digits[0]
                continue
            similar = False
            for q in range(p - 2, p + 3):
                if q == p or q not in cands:
                    continue
                for v in cands[q]:
                    if difflib.SequenceMatcher(None, core, _strip_digits(unit_text(v))).ratio() >= 0.6:
                        similar = True
            if similar:
                if digits and printed is None:
                    printed = digits[0]
            else:
                keep.append(u["uid"])
        info[p] = {"printed": printed, "keep": keep}
    return info

In [ ]:
#@title 3d · Code: page pipeline, cache, viewer (just run)
# ============================================================================
# PIPELINE: page -> layout -> routing -> OCR -> flags -> clean text -> cache
# ============================================================================
import base64
import html
import io
import json
import os
import time
from dataclasses import dataclass, field

import numpy as np
from PIL import Image


@dataclass
class RunConfig:
    routing: str = "auto"            # auto | urdu | arabic
    conf_flag: float = 0.5           # words whose weakest token has p < this get "[?]"
    flag_mark: str = "[?]"
    probe_tokens: int = 40           # tokens each model reads while deciding Urdu vs Arabic
    ambiguous_margin: float = 0.15   # below this the paragraph is re-split and routed line by line
    tokens_per_line: int = 90
    max_tokens: int = 1500
    segment: bool = True             # False = OCR the whole page as one image
    dpi: int = 300
    poem_format: str = "lines"       # lines | tab
    page_markers: bool = True
    review_flag_ratio: float = 0.08  # page goes to review if more than this share of words is flagged
    layout: LayoutConfig = field(default_factory=LayoutConfig)


def render_pdf_page(doc, page_no, dpi):
    """page_no is 1-based. Returns a grayscale numpy array."""
    page = doc[page_no - 1]
    pix = page.get_pixmap(dpi=dpi, colorspace="gray", alpha=False)
    return np.frombuffer(pix.samples, np.uint8).reshape(pix.height, pix.width).copy()


def _budget(u, layout, cfg):
    if u.kind in ("table", "page"):
        n = max(1, int((u.bbox[3] - u.bbox[1]) / max(1.0, layout.line_h * 0.8)))
    else:
        n = max(1, len(u.lines))
    return int(min(cfg.max_tokens * (2 if u.kind == "page" else 1), 40 + cfg.tokens_per_line * n))


def _unit_dict(u):
    return {"uid": u.uid, "kind": u.kind, "region": u.region, "bbox": [int(v) for v in u.bbox],
            "group": u.group, "n_lines": len(u.lines),
            "meta": {k: v for k, v in u.meta.items() if isinstance(v, (str, int, float, bool))}}


def _save_img(img, path, max_w=None, quality=88):
    if max_w and img.width > max_w:
        img = img.resize((max_w, int(img.height * max_w / img.width)))
    if path.endswith(".jpg"):
        img.convert("RGB").save(path, quality=quality)
    else:
        img.save(path)


def process_page(engine, gray, page_no, cfg, errors_dir=None, log=print):
    t0 = time.time()
    lay = segment_page(gray, cfg.layout)
    units = [u for u in lay.units]
    if cfg.segment and not units:                                   # blank page
        return {"page": page_no, "angle": round(lay.angle, 2), "units": [], "layout_notes": lay.notes + ["blank"],
                "n_words": 0, "n_flagged": 0, "review": {"needed": False, "reasons": []}, "corrected": None,
                "seconds": round(time.time() - t0, 1), "_thumb": draw_layout(lay, width=700)}
    if not cfg.segment or len(units) > 150:
        x0, y0, x1, y1 = lay.text_box
        units = [Unit("u001", "page", "body", (x0, y0, x1, y1), np.zeros(0, np.int64), [], "g1")]
        lay.units = units
    figures = [u for u in units if u.kind == "figure"]
    work = [u for u in units if u.kind != "figure"]
    imgs = {u.uid: render_unit(lay, u, cfg.layout) for u in work}
    script, why, margin, done = {}, {}, {}, {}

    # ---------------- routing
    if cfg.routing in ("urdu", "arabic"):
        for u in work:
            script[u.uid], why[u.uid], margin[u.uid] = cfg.routing, "fixed", 9.9
    else:
        def route(unit_list):
            groups = {}
            for u in unit_list:
                groups.setdefault(u.group, []).append(u)
            reps = []
            for g, us in groups.items():
                reps += sorted(us, key=lambda u: -(u.bbox[2] - u.bbox[0]) * (u.bbox[3] - u.bbox[1]))[:2]
            ims = [imgs[u.uid] for u in reps]
            ru = engine.generate(ims, "urdu", [cfg.probe_tokens] * len(ims))
            ra = engine.generate(ims, "arabic", [cfg.probe_tokens] * len(ims))
            probe = {u.uid: (a, b) for u, a, b in zip(reps, ru, ra)}
            for g, us in groups.items():
                d, notes = 0.0, []
                for u in us:
                    if u.uid in probe:
                        s, m, w = decide_script(*probe[u.uid])
                        d += m if s == "urdu" else -m
                        notes.append(w)
                s = "urdu" if d >= 0 else "arabic"
                for u in us:
                    script[u.uid], why[u.uid], margin[u.uid] = s, " ; ".join(notes), abs(d)
                    if u.uid in probe:                       # short text already fully read by the probe
                        r = probe[u.uid][0 if s == "urdu" else 1]
                        if r["finished"]:
                            done[u.uid] = r

        route(work)
        if cfg.ambiguous_margin > 0:
            new_work = []
            resplit = []
            for u in work:
                if (margin[u.uid] < cfg.ambiguous_margin and u.kind in ("prose", "centered")
                        and len(u.lines) >= 2):
                    parts = line_units(u)
                    for p in parts:
                        imgs[p.uid] = render_unit(lay, p, cfg.layout)
                    new_work += parts; resplit += parts
                    done.pop(u.uid, None)
                else:
                    new_work.append(u)
            if resplit:
                log(f"    {len({p.meta['parent'] for p in resplit})} mixed/unclear paragraph(s) re-read line by line")
                route(resplit)
                work = new_work
                units = [x for u in units for x in ([u] if u.kind == "figure" or u in work else
                                                    [p for p in resplit if p.meta["parent"] == u.uid])]

    # ---------------- full OCR per script
    results = dict(done)
    for s in ("urdu", "arabic"):
        todo = [u for u in work if script[u.uid] == s and u.uid not in results]
        if not todo:
            continue
        outs = engine.generate([imgs[u.uid] for u in todo], s, [_budget(u, lay, cfg) for u in todo])
        for u, r in zip(todo, outs):
            results[u.uid] = r
    # ---------------- repair loops / truncation
    for u in work:
        r = results[u.uid]
        issues = []
        if find_loop(r["text"]):
            r2 = engine.generate([imgs[u.uid]], script[u.uid], [_budget(u, lay, cfg)], repetition_penalty=1.15)[0]
            if not find_loop(r2["text"]):
                r2["retried"] = "loop"; results[u.uid] = r = r2
            else:
                issues.append("repetition loop")
        elif not r["finished"]:
            r2 = engine.generate([imgs[u.uid]], script[u.uid], [min(2 * _budget(u, lay, cfg), 2 * cfg.max_tokens)])[0]
            r2["retried"] = "length"; results[u.uid] = r = r2
            if not r["finished"]:
                issues.append("output cut off")
        r["issues"] = issues

    # ---------------- flags + clean-up
    rec_units, n_words, n_flag = [], 0, 0
    for u in units:
        d = _unit_dict(u)
        if u.kind == "figure":
            d.update(text="", script=None)
        else:
            r = results[u.uid]
            raw, looped = trim_loop(r["text"])
            spans = token_char_spans(engine.tok, r["ids"], r["text"]) if not looped else None
            flagged, nf, words = flag_uncertain(r["text"], spans, r["lp"], cfg.conf_flag, cfg.flag_mark)
            if looped:
                flagged = raw
            keep = u.kind in ("centered", "table", "page")
            d.update(script=script[u.uid], route=why[u.uid], margin=round(float(margin[u.uid]), 3),
                     raw=r["text"], text=clean_text(flagged, script[u.uid], keep_lines=keep),
                     mean_p=round(float(np.exp(r["mean_lp"])), 3), n_flagged=nf, n_words=len(words),
                     finished=bool(r["finished"]), issues=r.get("issues", []), retried=r.get("retried"))
            n_words += len(words); n_flag += nf
        rec_units.append(d)

    # ---------------- errors folder: tables, figures, and pages needing review
    reasons = []
    for d in rec_units:
        if d.get("issues"):
            reasons += [f"{d['uid']}: {x}" for x in d["issues"]]
        if d["kind"] in ("table", "figure"):
            reasons.append(f"{d['uid']}: {d['kind']}")
    if n_words and n_flag / n_words > cfg.review_flag_ratio:
        reasons.append(f"{n_flag}/{n_words} words flagged")
    unclear = [d["uid"] for d in rec_units if d.get("margin") is not None and d["margin"] < cfg.ambiguous_margin
               and cfg.routing == "auto"]
    if unclear:
        reasons.append(f"script unclear in {len(unclear)} unit(s)")
    if errors_dir:
        os.makedirs(errors_dir, exist_ok=True)
        full = Image.fromarray(lay.gray)
        for d in rec_units:
            if d["kind"] in ("table", "figure"):
                x0, y0, x1, y1 = d["bbox"]
                fn = f"p{page_no:04d}_{d['uid']}_{d['kind']}.png"
                _save_img(full.crop((x0, y0, x1, y1)), os.path.join(errors_dir, fn))
                d["file"] = "Errors/" + fn
        if reasons:
            _save_img(draw_layout(lay, {d["uid"]: d.get("script") for d in rec_units}, width=1400),
                      os.path.join(errors_dir, f"p{page_no:04d}_page.jpg"))
    return {"page": page_no, "angle": round(lay.angle, 2), "units": rec_units, "layout_notes": lay.notes,
            "n_words": n_words, "n_flagged": n_flag, "review": {"needed": bool(reasons), "reasons": reasons},
            "corrected": None, "seconds": round(time.time() - t0, 1),
            "_thumb": draw_layout(lay, {d["uid"]: d.get("script") for d in rec_units}, width=700)}


# ----------------------------------------------------------------------------- cache
def cache_path(chap_dir, page_no):
    return os.path.join(chap_dir, "_cache", f"page_{page_no:04d}.json")


def save_record(chap_dir, rec):
    p = cache_path(chap_dir, rec["page"])
    os.makedirs(os.path.dirname(p), exist_ok=True)
    data = {k: v for k, v in rec.items() if not k.startswith("_")}
    tmp = p + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=1)
    os.replace(tmp, p)


def load_record(chap_dir, page_no):
    p = cache_path(chap_dir, page_no)
    if not os.path.exists(p):
        return None
    with open(p, encoding="utf-8") as f:
        return json.load(f)


# ----------------------------------------------------------------------------- viewer
FONT_CSS = ("<link href='https://fonts.googleapis.com/css2?family=Noto+Nastaliq+Urdu&family=Amiri&display=swap' "
            "rel='stylesheet'>")


def _img_b64(img, q=80):
    buf = io.BytesIO()
    img.convert("RGB").save(buf, "JPEG", quality=q)
    return base64.b64encode(buf.getvalue()).decode()


def _esc_flags(t, mark):
    t = html.escape(t)
    m = html.escape(mark)
    return t.replace(m, f"<span style='background:#fde68a;color:#92400e;border-radius:3px;padding:0 2px'>{m}</span>")


def page_html(rec, thumb=None, mark="[?]", eta=""):
    units = rec["units"]
    n_u = sum(1 for u in units if u.get("script") == "urdu")
    n_a = sum(1 for u in units if u.get("script") == "arabic")
    rows = []
    for i, u in enumerate(units):
        s = u.get("script")
        font = "'Noto Nastaliq Urdu', serif; line-height:2.3" if s == "urdu" else "'Amiri', serif; line-height:1.9"
        badge = {"urdu": "#16a34a", "arabic": "#2563eb"}.get(s, "#6b7280")
        label = f"{i + 1} · {u['kind']}" + (f" · {s}" if s else "") + ("" if u["region"] == "body" else f" · {u['region']}")
        txt = u.get("corrected") if u.get("corrected") is not None else u.get("text", "")
        body = _esc_flags(txt, mark).replace("\n", "<br>") if txt else f"<i>{html.escape(u.get('file', ''))}</i>"
        warn = "".join(f"<span style='color:#b91c1c'> ⚠ {html.escape(x)}</span>" for x in u.get("issues", []))
        rows.append(f"<div style='margin:6px 0;padding:6px 8px;border-right:4px solid {badge};background:#fafafa'>"
                    f"<div dir='ltr' style='font:11px sans-serif;color:{badge}'>{html.escape(label)}{warn}</div>"
                    f"<div dir='rtl' style='font-family:{font};font-size:17px'>{body}</div></div>")
    rv = rec["review"]
    status = ("<span style='color:#b45309'>needs review: " + html.escape("; ".join(rv["reasons"][:4])) + "</span>"
              if rv["needed"] else "<span style='color:#15803d'>no issues flagged</span>")
    img = f"<img src='data:image/jpeg;base64,{_img_b64(thumb)}' style='width:100%;border:1px solid #ddd'>" if thumb else ""
    return (f"{FONT_CSS}<div style='font:13px sans-serif;margin-bottom:6px'><b>Page {rec['page']}</b> · "
            f"{rec['seconds']}s · {n_u} Urdu / {n_a} Arabic units · {rec['n_flagged']} of {rec['n_words']} words "
            f"flagged · {status} {html.escape(eta)}</div>"
            f"<div style='display:flex;gap:12px;align-items:flex-start'>"
            f"<div style='flex:0 0 42%'>{img}<div style='font:11px sans-serif;color:#555'>"
            f"<span style='color:#16a34a'>■</span> Urdu model &nbsp;<span style='color:#2563eb'>■</span> Arabic model "
            f"&nbsp;<span style='color:#ff8c00'>■</span> table &nbsp;<span style='color:#808080'>■</span> figure</div></div>"
            f"<div style='flex:1;max-height:900px;overflow:auto'>{''.join(rows)}</div></div>")


# ----------------------------------------------------------------------------- run + export
def run_pages(engine, pdf_path, pages, chap_dir, cfg, redo=(), show=True, log=print):
    try:
        import pymupdf as fitz
    except ImportError:
        import fitz
    doc = fitz.open(pdf_path)
    errors_dir = os.path.join(chap_dir, "Errors")
    todo = [p for p in pages if p in set(redo) or load_record(chap_dir, p) is None]
    skipped = len(pages) - len(todo)
    if skipped:
        log(f"{skipped} page(s) already done (cached) – resuming.")
    times = []
    for k, p in enumerate(todo):
        if not 1 <= p <= len(doc):
            log(f"page {p} is outside the PDF (1–{len(doc)}), skipped"); continue
        gray = render_pdf_page(doc, p, cfg.dpi)
        rec = process_page(engine, gray, p, cfg, errors_dir, log=log)
        save_record(chap_dir, rec)
        times.append(rec["seconds"])
        left = (len(todo) - k - 1) * float(np.mean(times[-5:]))
        eta = f"· {k + 1}/{len(todo)} done, ~{int(left // 60)} min left" if left else f"· {k + 1}/{len(todo)} done"
        if rec["review"]["needed"]:
            with open(os.path.join(errors_dir, "error_log.txt"), "a", encoding="utf-8") as f:
                f.write(f"page {p}\t" + "; ".join(rec["review"]["reasons"]) + f"\tErrors/p{p:04d}_page.jpg\n")
        if show:
            try:
                from IPython.display import HTML, clear_output, display
                clear_output(wait=True)
                display(HTML(page_html(rec, rec.get("_thumb"), cfg.flag_mark, eta)))
            except ImportError:
                pass
        log(f"page {p}: {rec['seconds']}s, {len(rec['units'])} units, {rec['n_flagged']} flagged {eta}")
    return todo


def page_final_text(rec, hdr, cfg):
    if rec.get("corrected") is not None:
        return rec["corrected"]
    r = dict(rec)
    r["units"] = [dict(u, region="body") if u["uid"] in hdr["keep"] else u for u in rec["units"]]
    return assemble_page(r, cfg.poem_format)


def export_chapter(chap_dir, chap_name, pages, cfg, log=print):
    recs = [load_record(chap_dir, p) for p in pages]
    missing = [p for p, r in zip(pages, recs) if r is None]
    if missing:
        raise RuntimeError(f"Pages not processed yet: {missing[:20]} – run the OCR cell first.")
    hdrs = resolve_headers(recs)
    out, raw = [], []
    for r in recs:
        h = hdrs[r["page"]]
        mark = f"[p. {r['page']}" + (f" | printed {h['printed']}" if h["printed"] else "") + "]"
        txt = page_final_text(r, h, cfg)
        out.append((mark + "\n" if cfg.page_markers else "") + txt)
        raw.append(mark + "\n" + "\n\n".join(f"<{u['uid']} {u['kind']} {u.get('script')}>\n{u.get('raw', '')}"
                                             for u in r["units"]))
    final = os.path.join(chap_dir, f"{chap_name}.txt")
    with open(final, "w", encoding="utf-8") as f:
        f.write("\n\n".join(out).strip() + "\n")
    with open(os.path.join(chap_dir, f"{chap_name}_raw.txt"), "w", encoding="utf-8") as f:
        f.write("\n\n".join(raw).strip() + "\n")
    flagged = sum(r["n_flagged"] for r in recs)
    review = [r["page"] for r in recs if r["review"]["needed"] and r.get("corrected") is None]
    log(f"Saved {final}  ({len(recs)} pages, {flagged} words marked {cfg.flag_mark}; "
        f"{len(review)} page(s) flagged and not reviewed: {review[:15]})")
    return final

In [ ]:
#@title 3e · Code: model loading, test page, review screen (just run)
# ============================================================================
# COLAB HELPERS: model loading, calibration, review widget
# ============================================================================
import os


def load_ocr_models(base_mode="4bit", log=print):
    """Load Qwen2-VL-2B once and attach both LoRA adapters ('urdu', 'arabic')."""
    import torch
    import transformers
    from packaging.version import Version
    from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

    if not torch.cuda.is_available():
        raise RuntimeError("No GPU. In Colab: Runtime ▸ Change runtime type ▸ pick a GPU (T4, L4 or A100), then re-run.")
    gpu = torch.cuda.get_device_name(0)
    bf16 = torch.cuda.is_bf16_supported()
    dt = torch.bfloat16 if bf16 else torch.float16
    dkey = "dtype" if Version(transformers.__version__) >= Version("4.56") else "torch_dtype"
    log(f"GPU: {gpu} · {'bfloat16' if bf16 else 'float16'} · transformers {transformers.__version__}")

    def load(repo):
        return Qwen2VLForConditionalGeneration.from_pretrained(
            repo, device_map="cuda:0", attn_implementation="sdpa", **{dkey: dt})

    base = BASE_4BIT if base_mode == "4bit" else BASE_FP16
    try:
        model = load(base)
    except Exception as e:
        if base != BASE_4BIT:
            raise
        log(f"4-bit base could not load ({type(e).__name__}: {e}); using the full-precision base instead.")
        base = BASE_FP16
        model = load(base)
    if base == BASE_4BIT and dt == torch.float16:
        try:
            import bitsandbytes as bnb
            for m in model.modules():
                if isinstance(m, bnb.nn.Linear4bit):
                    m.compute_dtype = torch.float16
        except ImportError:
            pass
    model.eval()
    log(f"Base model: {base}")
    processor = AutoProcessor.from_pretrained(BASE_FP16)
    import gc
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            load_lora_adapters(model, ADAPTERS, log=log)
        except Exception as e:
            if base != BASE_4BIT:
                raise
            # newer transformers + PEFT versions cannot attach LoRA to this pre-quantised 4-bit base
            log(f"LoRA could not attach to the 4-bit base ({type(e).__name__}); switching to the "
                f"full-precision base (needs ≈5 GB GPU memory)…")
            del model
            gc.collect(); torch.cuda.empty_cache()
            base = BASE_FP16
            model = load(base).eval()
            log(f"Base model: {base}")
            load_lora_adapters(model, ADAPTERS, log=log)
    log(f"GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
    return model, processor


def calibrate(engine, pdf_path, page_no, cfg, sizes=(48, 64, 88), max_units=6):
    """OCR the first few units of one page at several text sizes and show them side by side,
    together with the Urdu/Arabic decision for each unit."""
    import copy
    import html as H
    from IPython.display import HTML, display
    try:
        import pymupdf as fitz
    except ImportError:
        import fitz
    doc = fitz.open(pdf_path)
    gray = render_pdf_page(doc, page_no, cfg.dpi)
    cols = []
    for s in sizes:
        c = copy.deepcopy(cfg)
        c.layout.target_line_px = s
        lay = segment_page(gray, c.layout)
        keep = {u.uid for u in [u for u in lay.units if u.kind not in ("figure",)][:max_units]}
        lay_units = [u for u in lay.units if u.uid in keep]
        rec = _process_subset(engine, gray, page_no, c, keep)
        cols.append((s, rec))
    th = "".join(f"<th style='padding:4px'>line height {s}px</th>" for s, _ in cols)
    n = max(len(r["units"]) for _, r in cols)
    rows = []
    for i in range(n):
        tds = []
        for _, r in cols:
            if i < len(r["units"]):
                u = r["units"][i]
                font = "Noto Nastaliq Urdu" if u.get("script") == "urdu" else "Amiri"
                tds.append(f"<td style='vertical-align:top;padding:6px;border-top:1px solid #ddd'>"
                           f"<div dir='ltr' style='font:11px sans-serif;color:#666'>{u['uid']} · {u['kind']} · "
                           f"{u.get('script')} · mean p {u.get('mean_p')}<br>{H.escape(u.get('route', ''))}</div>"
                           f"<div dir='rtl' style=\"font-family:'{font}';font-size:16px;line-height:2\">"
                           f"{H.escape(u.get('text', ''))}</div></td>")
            else:
                tds.append("<td></td>")
        rows.append("<tr>" + "".join(tds) + "</tr>")
    display(HTML(FONT_CSS + f"<table style='width:100%;border-collapse:collapse'><tr>{th}</tr>{''.join(rows)}</table>"))
    display(HTML(f"<img src='data:image/jpeg;base64,{_img_b64(cols[0][1]['_thumb'])}' style='width:45%'>"))


def _process_subset(engine, gray, page_no, cfg, keep_uids):
    """process_page restricted to some units (for calibration)."""
    orig = segment_page

    def limited(g, c):
        lay = orig(g, c)
        lay.units = [u for u in lay.units if u.uid in keep_uids]
        return lay

    globals()["segment_page"] = limited
    try:
        return process_page(engine, gray, page_no, cfg, errors_dir=None, log=lambda *a: None)
    finally:
        globals()["segment_page"] = orig


def review_pages(pdf_path, chap_dir, pages, cfg, only_flagged=True):
    """Page-by-page editor: page image on the left, editable text on the right.
    'Save' stores your text for that page (it replaces the OCR text in the final export)."""
    import ipywidgets as W
    from IPython.display import HTML, display
    try:
        import pymupdf as fitz
    except ImportError:
        import fitz
    doc = fitz.open(pdf_path)
    recs = {p: load_record(chap_dir, p) for p in pages}
    recs = {p: r for p, r in recs.items() if r is not None}
    queue = [p for p in pages if p in recs and (not only_flagged or recs[p]["review"]["needed"])]
    if not queue:
        display(HTML("<b>Nothing to review</b> – no processed page was flagged. "
                     "Set ONLY_FLAGGED = False to open every page."))
        return
    hdr = resolve_headers([recs[p] for p in sorted(recs)])
    state = {"i": 0}
    css = W.HTML(FONT_CSS + "<style>.rtl-ed textarea{direction:rtl;font-family:'Noto Nastaliq Urdu','Amiri',serif;"
                 "font-size:17px;line-height:2.1}</style>")
    head = W.HTML()
    img = W.Image(format="jpeg", layout=W.Layout(width="100%"))
    ta = W.Textarea(layout=W.Layout(width="100%", height="760px"))
    ta.add_class("rtl-ed")
    b_prev, b_save, b_skip = W.Button(description="◀ Previous"), W.Button(description="Save & next ▶", button_style="success"), W.Button(description="Skip ▶")
    b_reset = W.Button(description="Discard my edits", button_style="warning")
    msg = W.HTML()

    def show():
        p = queue[state["i"]]
        r = recs[p]
        pix = doc[p - 1].get_pixmap(dpi=110)
        img.value = pix.tobytes("jpeg")
        reasons = "; ".join(r["review"]["reasons"]) or "–"
        edited = " · <b style='color:#15803d'>edited</b>" if r.get("corrected") is not None else ""
        head.value = (f"<b>Page {p}</b> ({state['i'] + 1} of {len(queue)}){edited}<br>"
                      f"<span style='color:#b45309'>{reasons}</span><br>"
                      f"<span style='color:#666'>Words marked {cfg.flag_mark} were uncertain. Delete the mark once checked.</span>")
        ta.value = page_final_text(r, hdr[p], cfg)
        msg.value = ""

    def save(_=None, advance=True):
        p = queue[state["i"]]
        recs[p]["corrected"] = ta.value
        save_record(chap_dir, recs[p])
        msg.value = f"<span style='color:#15803d'>Saved page {p}.</span>"
        if advance and state["i"] < len(queue) - 1:
            state["i"] += 1; show()

    def move(d):
        state["i"] = max(0, min(len(queue) - 1, state["i"] + d)); show()

    def reset(_):
        p = queue[state["i"]]
        recs[p]["corrected"] = None
        save_record(chap_dir, recs[p]); show()
        msg.value = f"Page {p} back to the OCR text."

    b_save.on_click(save)
    b_skip.on_click(lambda _: move(1))
    b_prev.on_click(lambda _: move(-1))
    b_reset.on_click(reset)
    show()
    display(W.VBox([css, head, W.HBox([W.Box([img], layout=W.Layout(width="45%")),
                                       W.Box([ta], layout=W.Layout(width="55%"))]),
                    W.HBox([b_prev, b_save, b_skip, b_reset]), msg]))

In [ ]:
#@title 3f · Code: apply settings (just run)
import os

def parse_pages(s):
    out = []
    for part in str(s).replace(" ", "").split(","):
        if not part:
            continue
        if "-" in part:
            a, b = part.split("-", 1)
            out += list(range(int(a), int(b) + 1))
        else:
            out.append(int(part))
    return out


def current_setup():
    cfg = RunConfig(routing=SCRIPT_ROUTING, conf_flag=float(FLAG_BELOW_CONFIDENCE),
                    poem_format=POETRY_FORMAT, page_markers=bool(PAGE_MARKERS))
    cfg.layout.target_line_px = int(TEXT_LINE_HEIGHT_PX)
    cfg.layout.columns = int(COLUMNS)
    chap_dir = os.path.join(OUTPUT_FOLDER, CHAP_NAME)
    os.makedirs(chap_dir, exist_ok=True)
    if int(END_PAGE) < int(START_PAGE):
        raise ValueError("END_PAGE is before START_PAGE")
    return cfg, chap_dir, list(range(int(START_PAGE), int(END_PAGE) + 1)), parse_pages(REDO_PAGES)

cfg, CHAP_DIR, PAGES, REDO = current_setup()
print(f"Chapter folder: {CHAP_DIR}  ·  pages {PAGES[0]}–{PAGES[-1]} ({len(PAGES)})")

In [ ]:
#@title 4 · Load the two OCR models (first run downloads ≈3 GB)
model, processor = load_ocr_models("4bit" if BASE_MODEL.startswith("4bit") else "fp16")
engine = OCREngine(model, processor, batch_size=int(BATCH_SIZE))
print("✓ Ready.")

### Step 5: test one page before the whole chapter
This cell reads the first few blocks of one page at three text sizes. Check two things:
* **Which column reads best.** Put that value in `TEXT_LINE_HEIGHT_PX` (step 1).
* **Whether each block's `urdu` / `arabic` label is right.** The small grey line shows why it was chosen. If the labels are wrong for your book, or the range is all one language, set `SCRIPT_ROUTING` to `urdu` or `arabic`.

In [ ]:
#@title 5 · Test page
TEST_PAGE = 13  #@param {type:"integer"}
cfg, CHAP_DIR, PAGES, REDO = current_setup()
calibrate(engine, PDF_PATH, int(TEST_PAGE), cfg, sizes=(48, 64, 88), max_units=6)

In [ ]:
#@title 6 · Run OCR on the chapter (resumes automatically)
cfg, CHAP_DIR, PAGES, REDO = current_setup()
assert "engine" in globals(), "Run step 4 first."
run_pages(engine, PDF_PATH, PAGES, CHAP_DIR, cfg, redo=REDO)
print("Finished. Next: step 7 (review), then step 8 (export).")

### Step 7: review
Only pages that were flagged are shown: tables, figures, many `[?]` marks, loops, cut-off text, or an unclear script. The page image is on the left and the text is on the right. Edit the text, remove `[?]` marks you have checked, and click **Save & next**. Your version replaces the OCR text for that page in the export. You can come back later; saved edits stay in Drive.

Crops of tables and figures, a marked-up image of every flagged page, and `error_log.txt` are in the chapter's `Errors/` folder.

In [ ]:
#@title 7 · Review flagged pages
ONLY_FLAGGED = True  #@param {type:"boolean"}
cfg, CHAP_DIR, PAGES, REDO = current_setup()
review_pages(PDF_PATH, CHAP_DIR, PAGES, cfg, only_flagged=ONLY_FLAGGED)

In [ ]:
#@title 8 · Export `{CHAP_NAME}.txt`
DELETE_CACHE_AFTER_EXPORT = False  #@param {type:"boolean"}
#@markdown The cache holds every page's OCR and your review edits. Delete it only when you are happy with the text.
import shutil
cfg, CHAP_DIR, PAGES, REDO = current_setup()
out_path = export_chapter(CHAP_DIR, CHAP_NAME, PAGES, cfg)
print(f"Also saved: {CHAP_NAME}_raw.txt (unedited model output, block by block, for checking).")
if DELETE_CACHE_AFTER_EXPORT:
    shutil.rmtree(os.path.join(CHAP_DIR, "_cache"), ignore_errors=True)
    print("Cache deleted.")